# Advanced LLM Planning Evaluation Notebook

Builds on the baseline `Results Analysis.ipynb` with process-level, efficiency,
and cross-domain metrics.

## Requirements
```bash
pip install pandas numpy matplotlib seaborn scipy plotly
```
All other imports are from the Python standard library.

In [2]:
# ============================================================
# CONFIGURATION  — edit this cell only
# ============================================================
import pathlib

# Notebook lives at  .../Benchmark Framework/analysis/notebooks/
# So two .parent steps reach the Benchmark Framework root.
_NB_DIR = pathlib.Path().resolve()
PROJECT_ROOT = _NB_DIR.parent.parent   # .../Benchmark Framework

# Override with an absolute path if auto-detection fails:
# PROJECT_ROOT = pathlib.Path(r"C:\Users\simo2\Desktop\AI_FOR_INDUSTRY\LLM_Benchmark\Benchmark Framework")

RESULTS_DIR   = PROJECT_ROOT / "outputs" / "parsed"
TASKS_DIR     = PROJECT_ROOT / "tasks"

MAX_FUZZY_DISTANCE = 2          # Levenshtein threshold for fuzzy action matching

COMPOSITE_WEIGHTS = {           # must sum to 1.0
    'fasr'            : 0.25,
    'iwsr'            : 0.20,
    'exec_ratio'      : 0.20,
    'one_minus_halluc': 0.20,
    'pas'             : 0.15,
}
COT_BONUS_WEIGHT = 0.05         # added when CoT data is available; rescaled to keep sum=1
BOOTSTRAP_N      = 1000         # resamples for confidence intervals

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RESULTS_DIR  : {RESULTS_DIR}  (exists={RESULTS_DIR.exists()})")
print(f"TASKS_DIR    : {TASKS_DIR}    (exists={TASKS_DIR.exists()})")

PROJECT_ROOT : C:\Users\simo2\Desktop\AI_FOR_INDUSTRY\LLM_Benchmark\Benchmark Framework
RESULTS_DIR  : C:\Users\simo2\Desktop\AI_FOR_INDUSTRY\LLM_Benchmark\Benchmark Framework\outputs\parsed  (exists=True)
TASKS_DIR    : C:\Users\simo2\Desktop\AI_FOR_INDUSTRY\LLM_Benchmark\Benchmark Framework\tasks    (exists=True)


In [3]:
import os, re, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from typing import Dict, List, Set, Tuple, Optional, Any
from collections import defaultdict

try:
    from scipy.stats import spearmanr
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    warnings.warn("scipy not found — Spearman correlation will use pandas .corr()")

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, message='.*tight_layout.*')

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})
print("Imports OK  |  scipy:", HAS_SCIPY, " plotly:", HAS_PLOTLY)

Imports OK  |  scipy: True  plotly: False


---
# Section 1 — Setup and Data Loading
Parses plan output files and PDDL domain/problem files into a unified DataFrame.

In [4]:
# ── General-purpose utilities ────────────────────────────────────────────────

def levenshtein(s1: str, s2: str) -> int:
    """Standard DP Levenshtein distance between two strings."""
    if s1 == s2:
        return 0
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1, 1):
        curr = [i]
        for j, c2 in enumerate(s2, 1):
            curr.append(min(prev[j] + 1, curr[j-1] + 1, prev[j-1] + (c1 != c2)))
        prev = curr
    return prev[-1]


_ACT_RE = re.compile(r'\(\s*([^\s()]+)((?:\s+[^\s()]+)*)\s*\)')

def parse_action(action_str: str) -> Tuple[Optional[str], List[str]]:
    """Parse '(action_name arg1 arg2 ...)' into (name, [args])."""
    m = _ACT_RE.match(action_str.strip())
    if not m:
        return None, []
    name = m.group(1).lower()
    args = m.group(2).strip().lower().split() if m.group(2).strip() else []
    return name, args


def extract_pddl_actions_from_text(text: str) -> List[str]:
    """Return all PDDL-style action strings from arbitrary text."""
    return re.findall(r'\([^()\n]+\)', text)

print("Helpers defined.")

Helpers defined.


In [5]:
# ── PDDL Domain Parser ───────────────────────────────────────────────────────

def parse_domain_pddl(domain_path: str) -> Dict:
    """
    Parse a PDDL domain file.

    Input : path to domain.pddl
    Output: dict with keys
        action_names  – set of lowercase action schema names
        predicates    – set of lowercase predicate names
        functions     – set of lowercase function/fluent names
        schemas       – dict  action_name -> {params, prec_raw, eff_raw}

    Detects: which tokens an LLM plan can legally use.
    """
    result = dict(action_names=set(), predicates=set(), functions=set(), schemas={})
    try:
        text = open(domain_path, encoding='utf-8', errors='replace').read()
        text = re.sub(r';[^\n]*', '', text).lower()

        # predicates
        pm = re.search(r'\(:predicates(.*?)\)(?=\s*\(:|\s*\))', text, re.DOTALL)
        if pm:
            result['predicates'] = set(re.findall(r'\(\s*([a-z][a-z0-9_-]*)', pm.group(1)))

        # functions / fluents
        fm = re.search(r'\(:functions(.*?)\)(?=\s*\(:|\s*\))', text, re.DOTALL)
        if fm:
            result['functions'] = set(re.findall(r'\(\s*([a-z][a-z0-9_-]*)', fm.group(1)))

        # action schemas
        for m in re.finditer(r'\(:action\s+([a-z][a-z0-9_-]*)', text):
            aname  = m.group(1)
            start  = m.start()
            # find end of this action block
            nxt = re.search(r'\(:action|\(:durative|\Z', text[start + 1:])
            block  = text[start: start + 1 + (nxt.start() if nxt else len(text))]

            result['action_names'].add(aname)

            params_m = re.search(r':parameters\s*\(([^)]*)\)', block)
            params   = re.findall(r'\?([a-z][a-z0-9_-]*)', params_m.group(1)) if params_m else []

            prec_pos = block.find(':precondition')
            eff_pos  = block.find(':effect')
            prec_raw = block[prec_pos + len(':precondition'):eff_pos].strip() if prec_pos != -1 and eff_pos != -1 else ''
            eff_raw  = block[eff_pos  + len(':effect'):].strip()              if eff_pos  != -1                    else ''

            result['schemas'][aname] = dict(params=params, prec_raw=prec_raw, eff_raw=eff_raw)

    except Exception as e:
        warnings.warn(f"[domain parser] {domain_path}: {e}")
    return result


def parse_problem_pddl(problem_path: str) -> Dict:
    """
    Parse a PDDL problem file.

    Input : path to instance .pddl
    Output: dict with keys
        objects      – set of lowercase object identifiers
        init_atoms   – set of tuples (predicate, *args) for propositional facts
        init_numeric – dict (function, *args) -> float for numeric inits

    Detects: which objects and facts exist in the initial state.
    """
    result = dict(objects=set(), init_atoms=set(), init_numeric={})
    try:
        text = open(problem_path, encoding='utf-8', errors='replace').read()
        text = re.sub(r';[^\n]*', '', text).lower()

        # objects
        om = re.search(r'\(:objects(.*?)\)(?=\s*\(:|\s*\))', text, re.DOTALL)
        if om:
            for tok in om.group(1).split():
                if not tok.startswith('-') and re.match(r'^[a-z][a-z0-9_-]*$', tok):
                    result['objects'].add(tok)

        # init
        im = re.search(r'\(:init(.*?)(?=\(:goal|\(:metric|\Z)', text, re.DOTALL)
        if im:
            init_txt = im.group(1)
            # numeric: (= (func args...) value)
            for nm in re.finditer(r'\(=\s*\(\s*([a-z][a-z0-9_-]*)([^)]*)\)\s*([\d.]+)\s*\)', init_txt):
                key = (nm.group(1),) + tuple(nm.group(2).strip().split())
                result['init_numeric'][key] = float(nm.group(3))
            # propositional: (pred arg...)
            for am in re.finditer(r'\(([a-z][a-z0-9_-]*(?:\s+[a-z0-9][a-z0-9_-]*)*)\)', init_txt):
                tokens = am.group(1).split()
                if tokens[0] not in {'=', 'not', 'and', 'or', 'increase', 'decrease'}:
                    result['init_atoms'].add(tuple(tokens))

    except Exception as e:
        warnings.warn(f"[problem parser] {problem_path}: {e}")
    return result

print("PDDL parsers defined.")

PDDL parsers defined.


In [6]:
# ── Extended Plan File Parser ─────────────────────────────────────────────────

def parse_result_file(file_path: str) -> Dict:
    """
    Parse a benchmark plan .txt file.

    Input : path to <instance>_plan.txt
    Output: dict with keys
        actions  – list of raw action strings
        metadata – dict of metadata fields
        cot_text – chain-of-thought text (empty string if absent)

    File format expected::
        <optional CoT reasoning text>
        (action1 arg ...) (action2 ...) ...
        --- Processing Metadata ---
        Domain: ...
        Plan Valid: True/False
        ...
    """
    actions, metadata, cot_text = [], {}, ''
    try:
        content = open(file_path, encoding='utf-8', errors='replace').read()
        parts = content.split('--- Processing Metadata ---')
        plan_block  = parts[0]
        meta_block  = parts[1] if len(parts) > 1 else ''

        # metadata
        for line in meta_block.strip().splitlines():
            if ':' in line:
                k, v = line.split(':', 1)
                metadata[k.strip()] = v.strip()

        # split CoT text from action lines in plan_block
        lines = plan_block.strip().splitlines()
        action_lines, pre_lines = [], []
        found_actions = False
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('(') and stripped.endswith(')'):
                found_actions = True
                action_lines.append(stripped)
            elif not found_actions:
                pre_lines.append(stripped)
            # lines after first action block that are not actions are ignored

        cot_text = '\n'.join(pre_lines)
        actions  = action_lines

        # some files have all actions on one line separated by spaces
        if not actions:
            actions = extract_pddl_actions_from_text(plan_block)

    except Exception as e:
        warnings.warn(f"[plan parser] {file_path}: {e}")
    return dict(actions=actions, metadata=metadata, cot_text=cot_text)

print("Plan file parser defined.")

Plan file parser defined.


In [7]:
# ── Index PDDL domain/problem files ──────────────────────────────────────────
# Builds domain_info[domain_name] and problem_info[(domain, difficulty, instance)]

domain_info: Dict[str, Dict]  = {}
problem_info: Dict[Tuple, Dict] = {}
pddl_difficulty_map: Dict[str, str] = {}   # (domain, instance_stem) -> difficulty

_DIFF_FOLDERS = {'easy', 'medium', 'hard'}

if TASKS_DIR.exists():
    for domain_dir in sorted(TASKS_DIR.iterdir()):
        if not domain_dir.is_dir() or domain_dir.name.startswith('_') or domain_dir.name == 'metadata':
            continue
        dname = domain_dir.name

        # domain.pddl
        dp = domain_dir / 'domain' / 'domain.pddl'
        if dp.exists():
            domain_info[dname] = parse_domain_pddl(str(dp))
            domain_info[dname]['path'] = str(dp)
        else:
            domain_info[dname] = dict(action_names=set(), predicates=set(),
                                      functions=set(), schemas={}, path=None)

        # problem instances per difficulty
        for diff in _DIFF_FOLDERS:
            diff_dir = domain_dir / diff
            if not diff_dir.exists():
                continue
            for pf in sorted(diff_dir.glob('*.pddl')):
                key = (dname, diff, pf.stem)
                problem_info[key] = parse_problem_pddl(str(pf))
                problem_info[key]['path']       = str(pf)
                problem_info[key]['difficulty'] = diff
                pddl_difficulty_map[(dname, pf.stem)] = diff

print(f"Domains indexed : {sorted(domain_info.keys())}")
print(f"Problem files   : {len(problem_info)}")

Domains indexed : ['block-grouping', 'expedition', 'fo-counters', 'fo-sailing', 'rover', 'settlersnumeric']
Problem files   : 120


In [ ]:
# ── Load all plan result files ────────────────────────────────────────────────
# Path layout:
#   parsed/<run_id>/<model_id>/<protocol_id>/<domain>/<difficulty>/<instance>.json
#   scored/<run_id>/<model_id>/<protocol_id>/<domain>/<difficulty>/<instance>.json
#
# Actions        → parsed JSON: attempts[i]["parsed_plan"]["actions"]
# CoT text       → parsed JSON: attempts[i]["parsed_plan"]["reasoning"]
# Valid, n_iters → scored JSON: solved, iterations_used

import json as _json

_DIFF_VALID  = {'easy', 'medium', 'hard'}
_SCORED_DIR  = PROJECT_ROOT / 'outputs' / 'scored'

# ── Read CoT flag per protocol from YAML files (no extra dependency needed) ───
_PROTOCOL_COT: Dict[str, bool] = {}
_PROTOCOLS_DIR = PROJECT_ROOT / 'protocols'
if _PROTOCOLS_DIR.exists():
    for _pyaml in sorted(_PROTOCOLS_DIR.glob('*.yaml')):
        try:
            _text = _pyaml.read_text(encoding='utf-8')
            _pid_m = re.search(r'^protocol_id\s*:\s*(\S+)', _text, re.MULTILINE)
            _cot_m = re.search(r'include_chain_of_thought\s*:\s*(\S+)', _text)
            if _pid_m and _cot_m:
                _PROTOCOL_COT[_pid_m.group(1)] = _cot_m.group(1).lower() == 'true'
        except Exception as _e:
            warnings.warn(f"[protocol loader] {_pyaml.name}: {_e}")
print("Protocol CoT flags:", _PROTOCOL_COT)


def _read_scored(run_id: str, model: str, protocol: str,
                 domain: str, difficulty: str, instance: str) -> Optional[Dict]:
    """Return the scored JSON dict, or None if the file is absent or unreadable."""
    p = _SCORED_DIR / run_id / model / protocol / domain / difficulty / f"{instance}.json"
    if not p.exists():
        return None
    try:
        return _json.loads(p.read_text(encoding='utf-8'))
    except Exception as e:
        warnings.warn(f"[scored read] {p}: {e}")
        return None


rows = []

if RESULTS_DIR.exists():
    for run_dir in sorted(RESULTS_DIR.iterdir()):
        if not run_dir.is_dir() or run_dir.name.startswith('.'):
            continue
        run_id = run_dir.name                         # e.g. 2025-06-01_12-00-00

        for model_dir in sorted(run_dir.iterdir()):
            if not model_dir.is_dir():
                continue
            model = model_dir.name                   # model_id from registry

            for protocol_dir in sorted(model_dir.iterdir()):
                if not protocol_dir.is_dir():
                    continue
                protocol     = protocol_dir.name     # e.g. direct_plan
                cot_protocol = _PROTOCOL_COT.get(protocol, False)

                for domain_dir in sorted(protocol_dir.iterdir()):
                    if not domain_dir.is_dir():
                        continue
                    domain = domain_dir.name         # task_family / domain name

                    for diff_dir in sorted(domain_dir.iterdir()):
                        if not diff_dir.is_dir() or diff_dir.name not in _DIFF_VALID:
                            continue
                        difficulty = diff_dir.name   # easy | medium | hard

                        for json_file in sorted(diff_dir.glob('*.json')):
                            instance = json_file.stem

                            try:
                                parsed_data = _json.loads(
                                    json_file.read_text(encoding='utf-8')
                                )
                            except Exception as e:
                                warnings.warn(f"[load] {json_file}: {e}")
                                continue

                            attempts = parsed_data.get('attempts', [])

                            # Pick the last attempt that has a non-empty action list
                            actions, cot_text = [], ''
                            for attempt in reversed(attempts):
                                pp        = attempt.get('parsed_plan') or {}
                                candidate = pp.get('actions') or []
                                if candidate:
                                    actions  = candidate
                                    cot_text = pp.get('reasoning', '')
                                    break

                            # Fallback: accept the very last attempt even if empty
                            if not actions and attempts:
                                pp       = attempts[-1].get('parsed_plan') or {}
                                actions  = pp.get('actions') or []
                                cot_text = pp.get('reasoning', '')

                            # Validity + iteration count from the scored artifact
                            scored = _read_scored(
                                run_id, model, protocol, domain, difficulty, instance
                            )
                            if scored is not None:
                                is_valid = bool(scored.get('solved', False))
                                n_iters  = scored.get('iterations_used', len(attempts))
                            else:
                                warnings.warn(
                                    f"[load] no scored file for "
                                    f"{run_id}/{model}/{protocol}/"
                                    f"{domain}/{difficulty}/{instance}"
                                )
                                is_valid = False
                                n_iters  = len(attempts)

                            rows.append(dict(
                                Model            = model,
                                Domain           = domain,
                                Problem          = instance,
                                Difficulty       = difficulty,
                                Protocol         = protocol,
                                Run_id           = run_id,
                                Valid            = is_valid,
                                Length           = len(actions),
                                Iterations       = n_iters,
                                Chain_of_Thought = cot_protocol or bool(cot_text.strip()),
                                _actions         = actions,
                                _cot_text        = cot_text,
                                _file_path       = str(json_file),
                            ))
else:
    warnings.warn(f"RESULTS_DIR does not exist: {RESULTS_DIR}. DataFrame will be empty.")

df = pd.DataFrame(rows)

if df.empty:
    print("⚠  No data found. All metric cells will skip computation but will not crash.")
    df = pd.DataFrame(columns=[
        'Model', 'Domain', 'Problem', 'Difficulty', 'Protocol', 'Run_id',
        'Valid', 'Length', 'Iterations', 'Chain_of_Thought',
        '_actions', '_cot_text', '_file_path',
    ])
else:
    df['Iterations'] = pd.to_numeric(df['Iterations'], errors='coerce')
    print(f"Loaded {len(df)} plan records")
    print(f"  Models    : {sorted(df['Model'].unique())}")
    print(f"  Domains   : {sorted(df['Domain'].unique())}")
    print(f"  Protocols : {sorted(df['Protocol'].unique())}")
    print(f"  Run IDs   : {sorted(df['Run_id'].unique())}")
    print(df[['Model', 'Domain', 'Protocol', 'Difficulty', 'Valid', 'Length', 'Iterations']].head())

df.head()

In [9]:
# ── Consistent color palette per model (used in every subsequent plot) ────────

ALL_MODELS = sorted(df['Model'].unique()) if not df.empty else []
_base_colors = sns.color_palette('tab20', n_colors=max(len(ALL_MODELS), 1))
MODEL_PALETTE = {m: _base_colors[i] for i, m in enumerate(ALL_MODELS)}

print("Model palette:", MODEL_PALETTE)

Model palette: {}


---
# Section 2 — Process-Level Metrics (Reasoning Quality)

## 2A — Action Hallucination Rate
Measures how often a model invents action names or objects not present in the PDDL domain/problem.

In [10]:
def compute_hallucination_metrics(actions: List[str],
                                  d_info: Dict,
                                  p_info: Dict) -> Dict:
    """
    Compute action and object hallucination rates for a single plan.

    Input:
        actions – list of raw action strings from the plan
        d_info  – domain info dict (from parse_domain_pddl)
        p_info  – problem info dict (from parse_problem_pddl)

    Output: dict with
        hallucinated_action_count, fuzzy_hallucinated_count,
        object_hallucination_count, total_action_count, total_arg_count,
        hallucination_rate, fuzzy_hallucination_rate, object_hallucination_rate

    Detects: invented action names and invented object references.
    """
    legal_actions = d_info.get('action_names', set())
    legal_objects = p_info.get('objects', set())

    hall_strict = hall_fuzzy = obj_hall = total_acts = total_args = 0

    for act_str in actions:
        aname, aargs = parse_action(act_str)
        if aname is None:
            continue
        total_acts += 1
        total_args += len(aargs)

        if aname not in legal_actions:
            hall_strict += 1
            # check fuzzy (nearest legal action within threshold)
            if not legal_actions or min(levenshtein(aname, la) for la in legal_actions) > MAX_FUZZY_DISTANCE:
                hall_fuzzy += 1

        for arg in aargs:
            if legal_objects and arg not in legal_objects:
                obj_hall += 1

    return dict(
        hallucinated_action_count   = hall_strict,
        fuzzy_hallucinated_count    = hall_fuzzy,
        object_hallucination_count  = obj_hall,
        total_action_count          = total_acts,
        total_arg_count             = total_args,
        hallucination_rate          = hall_strict / total_acts if total_acts else float('nan'),
        fuzzy_hallucination_rate    = hall_fuzzy  / total_acts if total_acts else float('nan'),
        object_hallucination_rate   = obj_hall    / total_args if total_args else float('nan'),
    )

print("compute_hallucination_metrics() defined.")

compute_hallucination_metrics() defined.


In [11]:
# ── Compute hallucination metrics for all rows ────────────────────────────────

halluc_rows = []
for _, row in df.iterrows():
    dom  = row['Domain']
    inst = row['Problem']
    diff = row.get('Difficulty', 'unknown')
    dinfo = domain_info.get(dom, {})
    pinfo = problem_info.get((dom, diff, inst), {})
    pddl_ok = bool(dinfo.get('action_names')) and bool(pinfo.get('objects'))

    try:
        hm = compute_hallucination_metrics(row['_actions'], dinfo, pinfo)
    except Exception as e:
        warnings.warn(f"Hallucination error {row['_file_path']}: {e}")
        hm = {k: float('nan') for k in ['hallucinated_action_count','fuzzy_hallucinated_count',
                                          'object_hallucination_count','total_action_count',
                                          'total_arg_count','hallucination_rate',
                                          'fuzzy_hallucination_rate','object_hallucination_rate']}
    hm['pddl_available'] = pddl_ok
    halluc_rows.append(hm)

halluc_df = pd.DataFrame(halluc_rows)
df = pd.concat([df.reset_index(drop=True), halluc_df.reset_index(drop=True)], axis=1)
print("Hallucination columns added:", halluc_df.columns.tolist())

Hallucination columns added: []


In [12]:
# ── Hallucination visualisations ──────────────────────────────────────────────
if df.empty or df['hallucination_rate'].isna().all():
    print("No hallucination data available.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Heatmap: Model × Domain → mean hallucination_rate
    pivot = df.pivot_table(values='hallucination_rate', index='Model', columns='Domain', aggfunc='mean')
    sns.heatmap(pivot, ax=axes[0], annot=True, fmt='.2f', cmap='YlOrRd', linewidths=.5, vmin=0, vmax=1)
    axes[0].set_title('Mean Hallucination Rate\n(Model × Domain)')
    axes[0].set_xlabel('Domain'); axes[0].set_ylabel('Model')

    # 2) Bar chart: Model → mean hallucination_rate, hue=Domain
    agg = df.groupby(['Model','Domain'])['hallucination_rate'].mean().reset_index()
    sns.barplot(data=agg, x='Model', y='hallucination_rate', hue='Domain', ax=axes[1], palette='Set2')
    axes[1].set_title('Hallucination Rate by Model & Domain')
    axes[1].set_xlabel('Model'); axes[1].set_ylabel('Hallucination Rate')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(axis='y', alpha=.4); axes[1].legend(fontsize=8)

    # 3) Object hallucination rate
    agg_obj = df.groupby(['Model','Domain'])['object_hallucination_rate'].mean().reset_index()
    sns.barplot(data=agg_obj, x='Model', y='object_hallucination_rate', hue='Domain', ax=axes[2], palette='Set2')
    axes[2].set_title('Object Hallucination Rate by Model & Domain')
    axes[2].set_xlabel('Model'); axes[2].set_ylabel('Object Hallucination Rate')
    axes[2].tick_params(axis='x', rotation=30)
    axes[2].grid(axis='y', alpha=.4); axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    # Strict vs fuzzy comparison table
    tbl = df.groupby('Model')[['hallucination_rate','fuzzy_hallucination_rate']].mean()
    tbl.columns = ['Strict Halluc Rate', 'Fuzzy Halluc Rate']
    tbl['Fuzzy-Reduction'] = tbl['Strict Halluc Rate'] - tbl['Fuzzy Halluc Rate']
    print("\nStrict vs Fuzzy Hallucination Rate:\n", tbl.round(4).to_string())

No hallucination data available.


## 2B — Precondition Awareness Score
Simulates plan execution step-by-step, classifying failures as *sequencing errors*
(wrong order, but the predicate existed earlier) or *state fabrications* (the predicate
was never established).

In [13]:
class PDDLSimulator:
    """
    Lightweight propositional + numeric PDDL simulator.

    Supports STRIPS-style propositional state and simple numeric fluents
    (increase/decrease with constant or single-fluent amounts).
    Complex numeric expressions (nested arithmetic in preconditions) are
    evaluated on a best-effort basis; failures are treated as state fabrications.
    """

    def __init__(self, d_info: Dict, p_info: Dict):
        self.d_info    = d_info
        self.prop      = set(p_info.get('init_atoms', set()))          # set of tuples
        self.num       = dict(p_info.get('init_numeric', {}))          # key-tuple -> float
        self.prop_hist = [frozenset(self.prop)]                        # history for temporal dist
        self.step      = 0

    # ── grounding ────────────────────────────────────────────────────────────
    def _ground(self, raw: str, params: List[str], args: List[str]) -> str:
        result = raw
        for p, a in zip(params, args):
            result = re.sub(r'\?' + re.escape(p) + r'\b', a, result)
        return result

    # ── numeric expression evaluator (best-effort) ────────────────────────────
    def _eval_num(self, expr: str) -> Optional[float]:
        expr = expr.strip()
        try:
            return float(expr)
        except ValueError:
            pass
        # (func arg ...)
        m = re.match(r'^\(\s*([a-z][a-z0-9_-]*)([^)]*)\)$', expr)
        if m:
            key = (m.group(1),) + tuple(m.group(2).strip().split())
            return self.num.get(key)
        # (op e1 e2) — find split at depth 0
        m2 = re.match(r'^\(\s*([+\-*/])\s(.+)\)$', expr, re.DOTALL)
        if m2:
            op, inner = m2.group(1), m2.group(2).strip()
            depth, pos = 0, 0
            operands = []
            start = 0
            for i, ch in enumerate(inner):
                if ch == '(':
                    depth += 1
                elif ch == ')':
                    depth -= 1
                elif ch == ' ' and depth == 0:
                    seg = inner[start:i].strip()
                    if seg:
                        operands.append(seg)
                    start = i + 1
            seg = inner[start:].strip()
            if seg:
                operands.append(seg)
            if len(operands) == 2:
                v1, v2 = self._eval_num(operands[0]), self._eval_num(operands[1])
                if v1 is not None and v2 is not None:
                    return {'+': v1+v2, '-': v1-v2, '*': v1*v2,
                             '/': v1/v2 if v2 else None}.get(op)
        return None

    # ── precondition checking ─────────────────────────────────────────────────
    def check_preconditions(self, prec_raw: str, params: List[str], args: List[str]):
        """
        Returns (satisfied: bool, failed_prop: list, failed_num: list).
        failed_prop holds tuple atoms that are missing from current state.
        failed_num  holds raw numeric condition strings that are violated.
        """
        grounded = self._ground(prec_raw, params, args)
        failed_prop, failed_num = [], []
        funcs = self.d_info.get('functions', set())
        skip_heads = {'and','or','not','>=','<=','>','<','=',
                      'increase','decrease','when','forall','exists',
                      '+','-','*','/'}

        # numeric comparisons
        for nm in re.finditer(r'\((>=|<=|>|<)\s*([^)]+)\s*([0-9.]+)\s*\)', grounded):
            op_s, lhs_s, rhs_s = nm.group(1), nm.group(2).strip(), nm.group(3)
            lhs = self._eval_num(lhs_s)
            rhs = float(rhs_s)
            if lhs is None:
                continue
            ok = {'>=': lhs>=rhs, '<=': lhs<=rhs, '>': lhs>rhs, '<': lhs<rhs}[op_s]
            if not ok:
                failed_num.append(nm.group(0))

        # propositional atoms
        no_neg = re.sub(r'\(not\s+\([^)]+\)\s*\)', '', grounded)
        for am in re.finditer(r'\(([a-z][a-z0-9_-]*)([^()]*)\)', no_neg):
            head = am.group(1)
            if head in skip_heads or head in funcs:
                continue
            atom = (head,) + tuple(am.group(2).strip().split()) if am.group(2).strip() else (head,)
            if atom not in self.prop:
                failed_prop.append(atom)

        sat = not failed_prop and not failed_num
        return sat, failed_prop, failed_num

    # ── effect application ────────────────────────────────────────────────────
    def apply_effects(self, eff_raw: str, params: List[str], args: List[str]):
        grounded = self._ground(eff_raw, params, args)
        funcs    = self.d_info.get('functions', set())
        skip_h   = {'and','or','not','>=','<=','>','<','=',
                    'increase','decrease','when','forall','assign','+','-','*','/'}

        # delete effects: (not (pred args))
        for dm in re.finditer(r'\(not\s+\(([a-z][a-z0-9_-]*)([^)]*)\)\s*\)', grounded):
            atom = (dm.group(1),) + tuple(dm.group(2).strip().split())
            self.prop.discard(atom)

        # add effects (ignore numeric ops)
        no_del = re.sub(r'\(not\s+\([^)]+\)\s*\)', '', grounded)
        for am in re.finditer(r'\(([a-z][a-z0-9_-]*)([^()]*)\)', no_del):
            head = am.group(1)
            if head in skip_h or head in funcs:
                continue
            atom = (head,) + tuple(am.group(2).strip().split()) if am.group(2).strip() else (head,)
            self.prop.add(atom)

        # numeric effects: (increase (func args) val) / (decrease ...)
        for op_name in ('increase', 'decrease'):
            for nm in re.finditer(
                    r'\(' + op_name + r'\s+\(\s*([a-z][a-z0-9_-]*)([^)]*)\)\s*([0-9.]+|\([^)]+\))\s*\)',
                    grounded):
                key = (nm.group(1),) + tuple(nm.group(2).strip().split())
                val = self._eval_num(nm.group(3).strip())
                if val is not None:
                    if op_name == 'increase':
                        self.num[key] = self.num.get(key, 0.0) + val
                    else:
                        self.num[key] = self.num.get(key, 0.0) - val

        self.prop_hist.append(frozenset(self.prop))
        self.step += 1

    # ── temporal history queries ──────────────────────────────────────────────
    def last_step_atom_true(self, atom) -> int:
        """Return the most recent step where atom held, or -1 if never."""
        for i in range(len(self.prop_hist) - 1, -1, -1):
            if atom in self.prop_hist[i]:
                return i
        return -1

print("PDDLSimulator defined.")

PDDLSimulator defined.


In [14]:
def compute_precondition_metrics(actions: List[str], d_info: Dict, p_info: Dict,
                                  label: str = '') -> Dict:
    """
    Simulate plan execution and compute precondition-awareness metrics.

    Input:
        actions – list of raw action strings
        d_info  – domain info dict
        p_info  – problem info dict
        label   – used only for warning messages

    Output: dict with
        executability_prefix_length, executability_ratio,
        sequencing_error_count, state_fabrication_count,
        precondition_awareness_score, mean_temporal_distance

    Classifies failures:
        sequencing_error   – precondition predicate was true at some earlier step
        state_fabrication  – predicate was never established in any prefix state
    """
    nan_result = dict(
        executability_prefix_length = 0,
        executability_ratio         = float('nan'),
        sequencing_error_count      = 0,
        state_fabrication_count     = 0,
        precondition_awareness_score= float('nan'),
        mean_temporal_distance      = float('nan'),
    )
    n = len(actions)
    if n == 0:
        return nan_result

    try:
        sim = PDDLSimulator(d_info, p_info)
        schemas = d_info.get('schemas', {})

        first_failure = n
        seq_errs, fab_errs = 0, 0
        temporal_dists: List[float] = []

        for step, act_str in enumerate(actions):
            aname, aargs = parse_action(act_str)
            if aname is None:
                continue
            schema = schemas.get(aname)
            if schema is None:
                continue  # unknown action — skip without failing

            params   = schema.get('params', [])
            prec_raw = schema.get('prec_raw', '')
            eff_raw  = schema.get('eff_raw', '')

            if len(params) != len(aargs):
                if first_failure == n:
                    first_failure = step
                fab_errs += 1
                break

            sat, failed_prop, failed_num = sim.check_preconditions(prec_raw, params, aargs)

            if not sat:
                if first_failure == n:
                    first_failure = step
                for atom in failed_prop:
                    last = sim.last_step_atom_true(atom)
                    if last >= 0:
                        seq_errs += 1
                        temporal_dists.append(step - last)
                    else:
                        fab_errs += 1
                fab_errs += len(failed_num)  # numeric failures → fabrication
                break

            sim.apply_effects(eff_raw, params, aargs)

        total_fail = seq_errs + fab_errs
        return dict(
            executability_prefix_length  = first_failure,
            executability_ratio          = first_failure / n,
            sequencing_error_count       = seq_errs,
            state_fabrication_count      = fab_errs,
            precondition_awareness_score = seq_errs / total_fail if total_fail > 0 else float('nan'),
            mean_temporal_distance       = float(np.mean(temporal_dists)) if temporal_dists else float('nan'),
        )
    except Exception as e:
        warnings.warn(f"[simulator] {label}: {e}")
        return nan_result

print("compute_precondition_metrics() defined.")

compute_precondition_metrics() defined.


In [15]:
prec_rows = []
for _, row in df.iterrows():
    dom  = row['Domain']
    inst = row['Problem']
    diff = row.get('Difficulty', 'unknown')
    dinfo = domain_info.get(dom, {})
    pinfo = problem_info.get((dom, diff, inst), {})
    pm = compute_precondition_metrics(row['_actions'], dinfo, pinfo, label=row['_file_path'])
    prec_rows.append(pm)

prec_df = pd.DataFrame(prec_rows)
df = pd.concat([df.reset_index(drop=True), prec_df.reset_index(drop=True)], axis=1)
print("Precondition columns added:", prec_df.columns.tolist())

Precondition columns added: []


In [16]:
if df.empty or df['executability_ratio'].isna().all():
    print("No precondition data available.")
else:
    # 1) Box plot: executability_ratio by Model & Domain
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.boxplot(data=df.dropna(subset=['executability_ratio']),
                x='Model', y='executability_ratio', hue='Domain',
                ax=axes[0], palette='Set2')
    axes[0].set_title('Executability Ratio by Model & Domain')
    axes[0].set_xlabel('Model'); axes[0].set_ylabel('Executability Ratio (steps before failure / plan length)')
    axes[0].tick_params(axis='x', rotation=30); axes[0].grid(axis='y', alpha=.4)

    # 2) Stacked bar: sequencing_error vs state_fabrication per model
    err_agg = df.groupby('Model')[['sequencing_error_count','state_fabrication_count']].sum()
    err_agg_pct = err_agg.div(err_agg.sum(axis=1), axis=0).fillna(0)
    err_agg_pct.plot(kind='bar', stacked=True, ax=axes[1],
                     color=['steelblue','tomato'], edgecolor='white')
    axes[1].set_title('Failure Type Breakdown per Model\n(proportion of total failures)')
    axes[1].set_xlabel('Model'); axes[1].set_ylabel('Proportion')
    axes[1].tick_params(axis='x', rotation=30); axes[1].grid(axis='y', alpha=.4)
    axes[1].legend(['Sequencing Error','State Fabrication'])
    plt.tight_layout(); plt.show()

    # 3) Scatter: executability_ratio vs plan length, coloured by model, faceted by domain
    doms = df['Domain'].dropna().unique()
    ncols = max(1, len(doms))
    fig2, axes2 = plt.subplots(1, ncols, figsize=(5*ncols, 4), squeeze=False)
    for ax, dom in zip(axes2[0], doms):
        sub = df[df['Domain'] == dom].dropna(subset=['executability_ratio','Length'])
        for model, grp in sub.groupby('Model'):
            ax.scatter(grp['Length'], grp['executability_ratio'],
                       label=model, alpha=.6, s=30,
                       color=MODEL_PALETTE.get(model, 'gray'))
        ax.set_title(f'Domain: {dom}'); ax.set_xlabel('Plan Length')
        ax.set_ylabel('Executability Ratio'); ax.grid(alpha=.3); ax.legend(fontsize=7)
    plt.suptitle('Executability Ratio vs Plan Length (by Domain)', y=1.01)
    plt.tight_layout(); plt.show()

    # 4) Histogram of temporal distance per model
    td_data = df.dropna(subset=['mean_temporal_distance'])
    if not td_data.empty:
        models_td = td_data['Model'].unique()
        nc = max(1, len(models_td))
        fig3, axes3 = plt.subplots(1, nc, figsize=(5*nc, 4), squeeze=False)
        for ax, model in zip(axes3[0], models_td):
            sub = td_data[td_data['Model'] == model]['mean_temporal_distance']
            ax.hist(sub.dropna(), bins=10, color=MODEL_PALETTE.get(model, 'steelblue'),
                    edgecolor='white', alpha=.8)
            ax.set_title(f'{model}'); ax.set_xlabel('Mean Temporal Distance')
            ax.set_ylabel('Count'); ax.grid(alpha=.3)
        plt.suptitle('Temporal Distance for Sequencing Errors (per Model)', y=1.02)
        plt.tight_layout(); plt.show()

No precondition data available.


## 2C — CoT–Plan Alignment Score
Measures how well the chain-of-thought reasoning reflects the actions and objects
that actually appear in the final plan.

In [17]:
def compute_cot_alignment(cot_text: str, actions: List[str],
                           d_info: Dict, p_info: Dict) -> Dict:
    """
    Compute CoT–plan alignment score for one plan.

    Input:
        cot_text – chain-of-thought reasoning text extracted from plan file
        actions  – list of raw action strings in the final plan
        d_info   – domain info dict
        p_info   – problem info dict

    Output: dict with
        cot_action_coverage, cot_object_coverage, cot_alignment_score

    Detects: whether the model's reasoning references the actions and objects
    it later uses, indicating genuine planning vs post-hoc generation.
    """
    legal_action_names = d_info.get('action_names', set())
    legal_objects      = p_info.get('objects', set())
    cot_lower          = cot_text.lower()
    cot_tokens         = set(re.findall(r'[a-z][a-z0-9_-]*', cot_lower))

    # plan action names and arguments
    plan_anames, plan_objs = set(), set()
    for act in actions:
        n, args = parse_action(act)
        if n:
            plan_anames.add(n)
            plan_objs.update(args)

    cot_act_mentioned = cot_tokens & legal_action_names
    cot_obj_mentioned = cot_tokens & legal_objects

    cot_action_cov = len(cot_act_mentioned & plan_anames) / max(len(plan_anames), 1)
    cot_object_cov = len(cot_obj_mentioned & plan_objs)  / max(len(plan_objs),   1)
    cot_align      = (cot_action_cov + cot_object_cov) / 2

    return dict(
        cot_action_coverage  = cot_action_cov,
        cot_object_coverage  = cot_object_cov,
        cot_alignment_score  = cot_align,
    )

print("compute_cot_alignment() defined.")

compute_cot_alignment() defined.


In [18]:
cot_rows = []
for _, row in df.iterrows():
    cot_flag = str(row.get('Chain_of_Thought', '')).lower() in ('true', '1', 'yes')
    if cot_flag:
        dinfo = domain_info.get(row['Domain'], {})
        diff  = row.get('Difficulty', 'unknown')
        pinfo = problem_info.get((row['Domain'], diff, row['Problem']), {})
        try:
            cm_ = compute_cot_alignment(row['_cot_text'], row['_actions'], dinfo, pinfo)
        except Exception as e:
            warnings.warn(f"CoT error {row['_file_path']}: {e}")
            cm_ = dict(cot_action_coverage=float('nan'),
                       cot_object_coverage=float('nan'),
                       cot_alignment_score=float('nan'))
    else:
        cm_ = dict(cot_action_coverage=float('nan'),
                   cot_object_coverage=float('nan'),
                   cot_alignment_score=float('nan'))
    cot_rows.append(cm_)

cot_df = pd.DataFrame(cot_rows)
df = pd.concat([df.reset_index(drop=True), cot_df.reset_index(drop=True)], axis=1)
print("CoT columns added:", cot_df.columns.tolist())

CoT columns added: []


In [19]:
if df.empty or df['cot_alignment_score'].isna().all():
    print("No CoT data available.")
else:
    cot_sub = df.dropna(subset=['cot_alignment_score'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Bar chart: mean cot_alignment_score per Model, hue=Domain
    agg = cot_sub.groupby(['Model','Domain'])['cot_alignment_score'].mean().reset_index()
    sns.barplot(data=agg, x='Model', y='cot_alignment_score', hue='Domain',
                ax=axes[0], palette='Set2')
    axes[0].set_title('Mean CoT Alignment Score by Model & Domain')
    axes[0].set_xlabel('Model'); axes[0].set_ylabel('CoT Alignment Score')
    axes[0].tick_params(axis='x', rotation=30); axes[0].grid(axis='y', alpha=.4)

    # 2) Side-by-side: Success Rate for CoT=True vs CoT=False per model
    df['_cot_flag'] = df['Chain_of_Thought'].apply(lambda x: str(x).lower() in ('true','1','yes'))
    cot_sr = df.groupby(['Model','_cot_flag'])['Valid'].mean().reset_index()
    cot_sr['CoT'] = cot_sr['_cot_flag'].map({True:'CoT=True', False:'CoT=False'})
    sns.barplot(data=cot_sr, x='Model', y='Valid', hue='CoT', ax=axes[1], palette='Set1')
    axes[1].set_title('Success Rate: CoT=True vs CoT=False')
    axes[1].set_xlabel('Model'); axes[1].set_ylabel('Success Rate')
    axes[1].tick_params(axis='x', rotation=30); axes[1].grid(axis='y', alpha=.4)

    # 3) Violin: cot_alignment_score for valid vs invalid plans per model
    cot_sub2 = cot_sub.copy()
    cot_sub2['Validity'] = cot_sub2['Valid'].map({True:'Valid', False:'Invalid'})
    if len(cot_sub2['Model'].unique()) > 0:
        sns.violinplot(data=cot_sub2, x='Model', y='cot_alignment_score', hue='Validity',
                       split=True, ax=axes[2], palette='Set1', inner='quart')
        axes[2].set_title('CoT Alignment Distribution:\nValid vs Invalid Plans')
        axes[2].set_xlabel('Model'); axes[2].set_ylabel('CoT Alignment Score')
        axes[2].tick_params(axis='x', rotation=30); axes[2].grid(axis='y', alpha=.4)

    plt.tight_layout(); plt.show()

No CoT data available.


---
# Section 3 — Efficiency Metrics

## 3A — First-Attempt Success Rate (FASR)
FASR = fraction of problems solved on the very first generation attempt (Iterations == 1).

In [20]:
if df.empty:
    print("No data.")
else:
    df['_iter1'] = (df['Valid'] == True) & (df['Iterations'] == 1)

    # Aggregate FASR and overall SR by (Model, Domain)
    fasr_agg = df.groupby(['Model','Domain']).agg(
        FASR        = ('_iter1', 'mean'),
        Success_Rate= ('Valid',  'mean'),
    ).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) FASR by Model, hue=Domain
    sns.barplot(data=fasr_agg, x='Model', y='FASR', hue='Domain', ax=axes[0], palette='Set2')
    axes[0].set_title('First-Attempt Success Rate (FASR) by Model & Domain')
    axes[0].set_xlabel('Model'); axes[0].set_ylabel('FASR')
    axes[0].tick_params(axis='x', rotation=30); axes[0].grid(axis='y', alpha=.4)

    # 2) FASR vs SR comparison (per model, averaged across domains)
    model_rates = df.groupby('Model').agg(FASR=('_iter1','mean'),
                                          SR  =('Valid', 'mean')).reset_index()
    x_pos = np.arange(len(model_rates))
    w = 0.35
    axes[1].bar(x_pos - w/2, model_rates['SR'],   w, label='Success Rate', color='steelblue')
    axes[1].bar(x_pos + w/2, model_rates['FASR'], w, label='FASR',         color='tomato')
    axes[1].set_xticks(x_pos); axes[1].set_xticklabels(model_rates['Model'], rotation=30)
    axes[1].set_title('Success Rate vs FASR (overall)'); axes[1].set_ylabel('Rate')
    axes[1].legend(); axes[1].grid(axis='y', alpha=.4)

    # 3) FASR by difficulty bin (easy / medium / hard)
    fasr_diff = df.groupby(['Model','Difficulty'])['_iter1'].mean().reset_index()
    fasr_diff.columns = ['Model','Difficulty','FASR']
    diff_order = ['easy','medium','hard','unknown']
    fasr_diff['Difficulty'] = pd.Categorical(fasr_diff['Difficulty'],
                                             categories=diff_order, ordered=True)
    fasr_diff = fasr_diff.dropna(subset=['FASR'])
    if not fasr_diff.empty:
        sns.barplot(data=fasr_diff, x='Difficulty', y='FASR', hue='Model',
                    ax=axes[2], palette=MODEL_PALETTE, order=diff_order)
        axes[2].set_title('FASR by Difficulty Bin')
        axes[2].set_xlabel('Difficulty'); axes[2].set_ylabel('FASR')
        axes[2].grid(axis='y', alpha=.4); axes[2].legend(fontsize=8)

    plt.tight_layout(); plt.show()
    print(fasr_agg.round(4).to_string(index=False))

No data.


## 3B — Iteration-Weighted Success Rate (IWSR)
Penalises models that need many retries:
`IWSR = mean(Valid / Iterations)` per (Model, Domain).

In [21]:
if df.empty:
    print("No data.")
else:
    df['_iwsr_contrib'] = df.apply(
        lambda r: (1.0 / r['Iterations']) if (r['Valid'] and pd.notna(r['Iterations']) and r['Iterations'] > 0) else 0.0,
        axis=1
    )

    iwsr_agg = df.groupby(['Model','Domain']).agg(
        IWSR        = ('_iwsr_contrib', 'mean'),
        FASR        = ('_iter1',        'mean'),
        Success_Rate= ('Valid',         'mean'),
    ).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) IWSR by Model, hue=Domain
    sns.barplot(data=iwsr_agg, x='Model', y='IWSR', hue='Domain', ax=axes[0], palette='Set2')
    axes[0].set_title('IWSR by Model & Domain')
    axes[0].set_xlabel('Model'); axes[0].set_ylabel('IWSR')
    axes[0].tick_params(axis='x', rotation=30); axes[0].grid(axis='y', alpha=.4)

    # 2) Triple comparison: SR vs FASR vs IWSR
    mc = df.groupby('Model').agg(SR=('Valid','mean'),
                                  FASR=('_iter1','mean'),
                                  IWSR=('_iwsr_contrib','mean')).reset_index()
    x  = np.arange(len(mc)); w = 0.25
    axes[1].bar(x - w,   mc['SR'],   w, label='Success Rate', color='steelblue')
    axes[1].bar(x,       mc['FASR'], w, label='FASR',         color='tomato')
    axes[1].bar(x + w,   mc['IWSR'], w, label='IWSR',         color='seagreen')
    axes[1].set_xticks(x); axes[1].set_xticklabels(mc['Model'], rotation=30)
    axes[1].set_title('SR vs FASR vs IWSR per Model'); axes[1].set_ylabel('Score')
    axes[1].legend(); axes[1].grid(axis='y', alpha=.4)

    # 3) Per-iteration conditional success probability: P(Valid | reached attempt k)
    max_iter = int(df['Iterations'].dropna().max()) if not df['Iterations'].dropna().empty else 0
    if max_iter > 0:
        for model, mdf in df.groupby('Model'):
            ks, probs = [], []
            for k in range(1, max_iter + 1):
                sub = mdf[mdf['Iterations'] >= k]
                if len(sub) == 0:
                    continue
                prob = sub[sub['Iterations'] == k]['Valid'].mean()
                ks.append(k); probs.append(prob)
            if ks:
                axes[2].plot(ks, probs, marker='o', label=model,
                             color=MODEL_PALETTE.get(model, 'gray'))
        axes[2].set_title('P(Valid | reached attempt k)\nper Model')
        axes[2].set_xlabel('Iteration k'); axes[2].set_ylabel('P(Valid)')
        axes[2].legend(fontsize=8); axes[2].grid(alpha=.4)

    plt.tight_layout(); plt.show()

No data.


## 3C — Retry Gap Analysis
`retry_gap = Success_Rate − FASR`. Large gaps indicate dependence on iterative repair.

In [22]:
if df.empty:
    print("No data.")
else:
    rg = df.groupby('Model').agg(
        SR  = ('Valid',          'mean'),
        FASR= ('_iter1',         'mean'),
        IWSR= ('_iwsr_contrib',  'mean'),
    ).reset_index()
    rg['retry_gap'] = rg['SR'] - rg['FASR']
    rg = rg.sort_values('retry_gap', ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1) Diverging bar chart: models sorted by retry_gap
    colors = ['tomato' if v > 0 else 'steelblue' for v in rg['retry_gap']]
    axes[0].barh(rg['Model'], rg['retry_gap'], color=colors, edgecolor='white')
    axes[0].axvline(0, color='black', linewidth=.8, linestyle='--')
    axes[0].set_title('Retry Gap (SR − FASR) per Model\n'
                       'Positive = depends on retries')
    axes[0].set_xlabel('Retry Gap'); axes[0].set_ylabel('Model')
    axes[0].grid(axis='x', alpha=.4)

    # 2) Scatter: FASR (x) vs IWSR (y), size=SR, color=Domain-average SR
    # aggregate per model across all domains
    sc_data = df.groupby('Model').agg(
        FASR= ('_iter1',        'mean'),
        IWSR= ('_iwsr_contrib', 'mean'),
        SR  = ('Valid',         'mean'),
    ).reset_index()
    sc = axes[1].scatter(sc_data['FASR'], sc_data['IWSR'],
                          s=sc_data['SR'] * 1500 + 50,
                          c=sc_data['SR'], cmap='RdYlGn', vmin=0, vmax=1,
                          edgecolors='gray', linewidths=.5, alpha=.85)
    for _, r in sc_data.iterrows():
        axes[1].annotate(r['Model'], (r['FASR'], r['IWSR']),
                          textcoords='offset points', xytext=(6, 4), fontsize=8)
    plt.colorbar(sc, ax=axes[1], label='Overall Success Rate')
    axes[1].set_title('FASR vs IWSR\n(size = Success Rate)')
    axes[1].set_xlabel('FASR'); axes[1].set_ylabel('IWSR')
    axes[1].grid(alpha=.4)

    plt.tight_layout(); plt.show()
    print(rg[['Model','SR','FASR','IWSR','retry_gap']].round(4).to_string(index=False))

No data.


---
# Section 4 — Cross-Domain and Cross-Model Comparative Analysis

## 4A — Within-Domain Model Ranking
Rank models on each metric within each domain.

In [23]:
if df.empty:
    print("No data.")
else:
    RANK_METRICS = {
        'Success_Rate'    : 'max',  # higher is better
        'FASR'            : 'max',
        'IWSR'            : 'max',
        'exec_ratio'      : 'max',
        'hallucination_rate': 'min',   # lower is better
        'prec_awareness'  : 'max',
        'cot_alignment'   : 'max',
    }

    # Build aggregated metric table (Model, Domain) → scalar values
    per_dom = df.groupby(['Model','Domain']).agg(
        Success_Rate   = ('Valid',                    'mean'),
        exec_ratio     = ('executability_ratio',      'mean'),
        hallucination_rate = ('hallucination_rate',   'mean'),
        prec_awareness = ('precondition_awareness_score', lambda x: x.dropna().mean()),
        cot_alignment  = ('cot_alignment_score',      lambda x: x.dropna().mean()),
    ).reset_index()

    # Merge in FASR & IWSR
    per_dom = per_dom.merge(
        df.groupby(['Model','Domain']).agg(
            FASR=('_iter1',       'mean'),
            IWSR=('_iwsr_contrib','mean'),
        ).reset_index(),
        on=['Model','Domain'], how='left'
    )

    metric_cols = list(RANK_METRICS.keys())

    for dom in sorted(per_dom['Domain'].unique()):
        sub = per_dom[per_dom['Domain'] == dom].copy().set_index('Model')
        sub = sub[metric_cols]

        # Normalize to [0,1]
        normed = sub.copy()
        for col, direction in RANK_METRICS.items():
            if col not in normed.columns:
                normed[col] = float('nan')
                continue
            cmin, cmax = normed[col].min(), normed[col].max()
            if cmax > cmin:
                normed[col] = (normed[col] - cmin) / (cmax - cmin)
                if direction == 'min':
                    normed[col] = 1 - normed[col]
            else:
                normed[col] = 0.5

        # Heatmap
        fig, ax = plt.subplots(figsize=(max(10, len(metric_cols)*1.5), max(3, len(sub)*0.6+1)))
        sns.heatmap(normed.fillna(0).astype(float), ax=ax, annot=True, fmt='.2f',
                    cmap='RdYlGn', vmin=0, vmax=1, linewidths=.5)
        ax.set_title(f'Normalised Metric Scores — Domain: {dom}\n'
                     f'(1 = best in domain, 0 = worst)')
        ax.set_xlabel('Metric'); ax.set_ylabel('Model')
        plt.tight_layout(); plt.show()

        # Rank table
        rank_tbl = sub.rank(ascending=False, method='min').fillna('-')
        print(f"\nRank table for domain '{dom}' (1=best):")
        print(rank_tbl.round(0).to_string())

No data.


## 4B — Cross-Domain Consistency
Models with low rank variance are *generalist planners*; high variance indicates domain specialisation.

In [24]:
if df.empty:
    print("No data.")
else:
    sr_pivot = df.groupby(['Model','Domain'])['Valid'].mean().unstack('Domain').fillna(0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Heatmap: Model × Domain success rate (annotated)
    sns.heatmap(sr_pivot, ax=axes[0], annot=True, fmt='.2f', cmap='YlGn',
                linewidths=.4, vmin=0, vmax=1)
    axes[0].set_title('Success Rate Heatmap (Model × Domain)')
    axes[0].set_xlabel('Domain'); axes[0].set_ylabel('Model')

    # 2) Rank variance per model
    sr_ranks = sr_pivot.rank(ascending=False, method='min')
    rank_var = sr_ranks.var(axis=1).reset_index()
    rank_var.columns = ['Model', 'rank_variance']
    rank_var = rank_var.sort_values('rank_variance')
    axes[1].barh(rank_var['Model'], rank_var['rank_variance'],
                 color=[MODEL_PALETTE.get(m, 'steelblue') for m in rank_var['Model']],
                 edgecolor='white')
    axes[1].set_title('Rank Variance Across Domains\n(lower = more consistent)')
    axes[1].set_xlabel('Variance of Domain-Rank'); axes[1].set_ylabel('Model')
    axes[1].grid(axis='x', alpha=.4)

    # 3) Spearman correlation between domains
    if sr_pivot.shape[1] >= 2:
        if HAS_SCIPY:
            cols = sr_pivot.columns.tolist()
            corr_mat = np.zeros((len(cols), len(cols)))
            for i, c1 in enumerate(cols):
                for j, c2 in enumerate(cols):
                    r, _ = spearmanr(sr_pivot[c1], sr_pivot[c2])
                    corr_mat[i, j] = r
            corr_df = pd.DataFrame(corr_mat, index=cols, columns=cols)
        else:
            corr_df = sr_pivot.corr(method='spearman')

        sns.heatmap(corr_df, ax=axes[2], annot=True, fmt='.2f',
                    cmap='coolwarm', vmin=-1, vmax=1, linewidths=.4)
        axes[2].set_title('Spearman Correlation of Model\nSuccess Rates Across Domains')

    plt.tight_layout(); plt.show()

No data.


## 4C — Failure Mode Taxonomy Plot
Key interpretive plot: each model is positioned by *how* it fails.

In [25]:
if df.empty:
    print("No data.")
else:
    tax = df.groupby('Model').agg(
        halluc_rate  = ('hallucination_rate',          'mean'),
        pas          = ('precondition_awareness_score', lambda x: x.dropna().mean()),
        fasr         = ('_iter1',                      'mean'),
        sr           = ('Valid',                       'mean'),
    ).reset_index().fillna({'pas': 0.5})  # default PAS when no failures

    fig, ax = plt.subplots(figsize=(10, 8))

    sc = ax.scatter(
        tax['halluc_rate'], tax['pas'],
        s   = tax['fasr'] * 1200 + 80,
        c   = tax['sr'],
        cmap='RdYlGn', vmin=0, vmax=1,
        edgecolors='dimgray', linewidths=.8, alpha=.9, zorder=3
    )
    plt.colorbar(sc, ax=ax, label='Overall Success Rate')

    for _, r in tax.iterrows():
        ax.annotate(r['Model'], (r['halluc_rate'], r['pas']),
                    textcoords='offset points', xytext=(8, 4), fontsize=9, zorder=4)

    # Quadrant lines at midpoints
    xm = tax['halluc_rate'].mean() if len(tax) else 0.5
    ym = 0.5
    ax.axvline(xm, color='gray', linestyle='--', linewidth=.8, zorder=1)
    ax.axhline(ym, color='gray', linestyle='--', linewidth=.8, zorder=1)

    # Quadrant labels
    xl, xr = ax.get_xlim(); yb, yt = ax.get_ylim()
    pad = 0.01
    ax.text(xl + pad, yt - pad, 'Good vocabulary,\nordering problems',
            ha='left', va='top', fontsize=8, color='steelblue',
            bbox=dict(boxstyle='round,pad=.3', fc='lightyellow', ec='gray', alpha=.8))
    ax.text(xr - pad, yt - pad, 'Invents actions,\nhas causal intuition',
            ha='right', va='top', fontsize=8, color='tomato',
            bbox=dict(boxstyle='round,pad=.3', fc='lightyellow', ec='gray', alpha=.8))
    ax.text(xl + pad, yb + pad, 'Near-random output',
            ha='left', va='bottom', fontsize=8, color='purple',
            bbox=dict(boxstyle='round,pad=.3', fc='lightyellow', ec='gray', alpha=.8))
    ax.text(xr - pad, yb + pad, 'Knows words,\nlacks world model',
            ha='right', va='bottom', fontsize=8, color='darkgreen',
            bbox=dict(boxstyle='round,pad=.3', fc='lightyellow', ec='gray', alpha=.8))

    ax.set_xlabel('Hallucination Rate (higher → more grounding failures)', fontsize=11)
    ax.set_ylabel('Precondition Awareness Score\n(higher → more sequencing errors, fewer fabrications)', fontsize=11)
    ax.set_title('Failure Mode Taxonomy\n(dot size = FASR, colour = overall Success Rate)', fontsize=13)
    ax.grid(alpha=.3); plt.tight_layout(); plt.show()

No data.


---
# Section 5 — Composite Planning Score (PS)
$$PS = 0.25\cdot FASR + 0.20\cdot IWSR + 0.20\cdot\overline{exec\_ratio}
+ 0.20\cdot(1-\overline{halluc}) + 0.15\cdot\overline{PAS}$$
Optional CoT bonus adds $0.05\cdot cot\_alignment$ (renormalised so weights sum to 1).

In [26]:
def compute_composite_score(model_stats: Dict, weights: Dict,
                             cot_bonus_w: float = 0.05) -> float:
    """
    Compute the Composite Planning Score for one model.

    Input:
        model_stats   – dict with keys: fasr, iwsr, exec_ratio,
                          one_minus_halluc, pas, cot_alignment (optional, may be nan)
        weights       – COMPOSITE_WEIGHTS dict (must sum to 1.0)
        cot_bonus_w   – extra weight for CoT alignment; only applied if not nan

    Output: float PS in [0, 1]

    Detects: overall planning quality accounting for all dimensions.
    """
    base = (
        weights['fasr']             * model_stats.get('fasr', 0) +
        weights['iwsr']             * model_stats.get('iwsr', 0) +
        weights['exec_ratio']       * model_stats.get('exec_ratio', 0) +
        weights['one_minus_halluc'] * model_stats.get('one_minus_halluc', 0) +
        weights['pas']              * model_stats.get('pas', 0.5)
    )
    cot = model_stats.get('cot_alignment', float('nan'))
    if not math.isnan(cot):
        # renormalise so total = 1.0
        total_w = sum(weights.values()) + cot_bonus_w
        base = (base + cot_bonus_w * cot) / total_w
    return float(np.clip(base, 0, 1))

print("compute_composite_score() defined.")

compute_composite_score() defined.


In [27]:
if df.empty:
    print("No data.")
else:
    # Per-(Model, Domain)
    ps_dom = df.groupby(['Model','Domain']).agg(
        fasr              = ('_iter1',                      'mean'),
        iwsr              = ('_iwsr_contrib',               'mean'),
        exec_ratio        = ('executability_ratio',         'mean'),
        halluc            = ('hallucination_rate',          'mean'),
        pas               = ('precondition_awareness_score',lambda x: x.dropna().mean()),
        cot_alignment     = ('cot_alignment_score',         lambda x: x.dropna().mean()),
    ).reset_index()

    ps_dom['one_minus_halluc'] = 1 - ps_dom['halluc'].fillna(0)
    ps_dom['pas']              = ps_dom['pas'].fillna(0.5)

    ps_dom['PS'] = ps_dom.apply(
        lambda r: compute_composite_score(r.to_dict(), COMPOSITE_WEIGHTS, COT_BONUS_WEIGHT),
        axis=1
    )

    # Overall PS per model (mean across domains)
    ps_overall = ps_dom.groupby('Model')['PS'].mean().sort_values(ascending=False).reset_index()
    ps_overall.columns = ['Model','PS_overall']

    # Bootstrap CI
    rng = np.random.default_rng(seed=42)
    ci_data = []
    for model in ps_overall['Model']:
        contribs = df[df['Model'] == model].apply(
            lambda r: compute_composite_score(dict(
                fasr              = float(r.get('_iter1',           0) or 0),
                iwsr              = float(r.get('_iwsr_contrib',    0) or 0),
                exec_ratio        = float(r.get('executability_ratio', 0) or 0) if not pd.isna(r.get('executability_ratio', 0)) else 0,
                one_minus_halluc  = 1 - float(r.get('hallucination_rate', 0) or 0) if not pd.isna(r.get('hallucination_rate', 0)) else 0.5,
                pas               = float(r.get('precondition_awareness_score', 0.5) or 0.5) if not pd.isna(r.get('precondition_awareness_score', 0.5)) else 0.5,
                cot_alignment     = float(r.get('cot_alignment_score', float('nan')) or float('nan')),
            ), COMPOSITE_WEIGHTS, COT_BONUS_WEIGHT),
            axis=1
        ).values
        if len(contribs) == 0:
            ci_data.append({'Model': model, 'PS_ci_lo': 0, 'PS_ci_hi': 0})
            continue
        boots = [contribs[rng.integers(0, len(contribs), len(contribs))].mean()
                 for _ in range(BOOTSTRAP_N)]
        ci_data.append({
            'Model'   : model,
            'PS_ci_lo': np.percentile(boots, 2.5),
            'PS_ci_hi': np.percentile(boots, 97.5),
        })

    ci_df = pd.DataFrame(ci_data)
    ps_overall = ps_overall.merge(ci_df, on='Model')
    ps_overall['err_lo'] = ps_overall['PS_overall'] - ps_overall['PS_ci_lo']
    ps_overall['err_hi'] = ps_overall['PS_ci_hi'] - ps_overall['PS_overall']

    print("Composite Planning Scores (descending):")
    print(ps_overall[['Model','PS_overall','PS_ci_lo','PS_ci_hi']].round(4).to_string(index=False))

No data.


In [28]:
if df.empty or 'ps_overall' not in dir():
    print("No PS data.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # 1) Horizontal bar chart with CI error bars, coloured by domain contribution (stacked)
    pivot_ps = ps_dom.pivot_table(values='PS', index='Model', columns='Domain').fillna(0)
    pivot_ps_sorted = pivot_ps.loc[ps_overall['Model']]
    pivot_ps_sorted.plot(kind='barh', stacked=True, ax=axes[0],
                          colormap='Set2', edgecolor='white', alpha=.9)
    # overlay CI
    for i, (_, row) in enumerate(ps_overall.iterrows()):
        axes[0].errorbar(row['PS_overall'], i,
                         xerr=[[row['err_lo']], [row['err_hi']]],
                         fmt='none', color='black', capsize=4, linewidth=1.5)
    axes[0].set_title('Composite Planning Score (PS)\nby Model (stacked = domain contribution)')
    axes[0].set_xlabel('PS'); axes[0].set_ylabel('Model')
    axes[0].grid(axis='x', alpha=.4); axes[0].legend(title='Domain', fontsize=8, loc='lower right')

    # 2) Summary table
    axes[1].axis('off')
    tbl_data = ps_dom.groupby('Model').agg(
        FASR  = ('fasr',          'mean'),
        IWSR  = ('iwsr',          'mean'),
        Exec  = ('exec_ratio',    'mean'),
        Halluc= ('halluc',        'mean'),
        PAS   = ('pas',           'mean'),
        PS    = ('PS',            'mean'),
    ).round(3).sort_values('PS', ascending=False).reset_index()
    tbl_data.columns = ['Model','FASR','IWSR','Exec','Halluc','PAS','PS']
    tbl = axes[1].table(
        cellText    = tbl_data.values,
        colLabels   = tbl_data.columns,
        cellLoc     = 'center', loc='center'
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.2, 1.6)
    axes[1].set_title('Composite Score Summary Table', pad=20)

    plt.tight_layout(); plt.show()

    # 3) Radar / spider chart (matplotlib fallback)
    radar_metrics = ['FASR','IWSR','Exec','1-Halluc','PAS']
    N = len(radar_metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig3, ax3 = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for _, row in tbl_data.iterrows():
        vals = [row['FASR'], row['IWSR'], row['Exec'], 1 - row['Halluc'], row['PAS']]
        vals += vals[:1]
        ax3.plot(angles, vals, linewidth=1.5, label=row['Model'],
                 color=MODEL_PALETTE.get(row['Model'], 'gray'))
        ax3.fill(angles, vals, alpha=.08, color=MODEL_PALETTE.get(row['Model'], 'gray'))
    ax3.set_xticks(angles[:-1]); ax3.set_xticklabels(radar_metrics, fontsize=11)
    ax3.set_ylim(0, 1)
    ax3.set_title('Radar Chart: Component Metrics per Model', fontsize=13, pad=20)
    ax3.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
    ax3.grid(alpha=.4); plt.tight_layout(); plt.show()

No PS data.


---
# Section 6 — Summary and Interpretation Template

## Results Section Template (fill `{}` placeholders from the tables above)

### 6.1 Main Findings Paragraph

> We evaluated `{num_models}` LLM families on `{num_domains}` PDDL planning domains using
> the Composite Planning Score (PS), which integrates first-attempt success (FASR), iteration
> efficiency (IWSR), plan executability, action grounding accuracy, and causal reasoning quality.
> **`{best_model}`** achieved the highest overall PS of `{best_ps:.3f}` (95% CI:
> [`{best_ci_lo:.3f}`, `{best_ci_hi:.3f}`]), while **`{worst_model}`** scored the lowest at
> `{worst_ps:.3f}`. Models with the largest retry gap — particularly **`{highest_retry_gap_model}`**
> ($\\Delta = {retry_gap_val:.2f}$) — demonstrate a strong dependency on iterative repair rather
> than first-attempt correctness. In terms of cross-domain consistency, **`{most_consistent_model}`**
> exhibited the lowest rank variance ($\\sigma^2 = {min_rank_var:.2f}$), indicating robust
> performance across planning domains, whereas **`{least_consistent_model}`** showed the highest
> specialisation ($\\sigma^2 = {max_rank_var:.2f}$).

---

### 6.2 Formal Metric Definitions

Let $\\mathcal{M}$ be the set of models, $\\mathcal{D}$ the set of domains, and $\\mathcal{P}_{d}$
the set of problem instances for domain $d$.  For model $m$ on problem $p$ in domain $d$, let
$V_{m,p}\\in\\{0,1\\}$ indicate validity, $I_{m,p}$ the number of iterations used, and
$|\\pi_{m,p}|$ the plan length.

$$\\text{FASR}_{m,d} = \\frac{\\sum_{p\\in\\mathcal{P}_d} \\mathbf{1}[V_{m,p}=1 \\wedge I_{m,p}=1]}{|\\mathcal{P}_d|}$$

$$\\text{IWSR}_{m,d} = \\frac{1}{|\\mathcal{P}_d|}\\sum_{p\\in\\mathcal{P}_d}\\frac{V_{m,p}}{I_{m,p}}$$

$$\\text{Hallucination Rate}_{m,p} = \\frac{|\\{a \\in \\pi_{m,p} : \\text{name}(a) \\notin \\mathcal{A}_d\\}|}{|\\pi_{m,p}|}$$

where $\\mathcal{A}_d$ is the set of legal action schema names in domain $d$.

$$\\text{PS}_m = 0.25\\cdot\\overline{\\text{FASR}} + 0.20\\cdot\\overline{\\text{IWSR}}
+ 0.20\\cdot\\overline{\\text{exec\\_ratio}} + 0.20\\cdot(1-\\overline{\\text{halluc}})
+ 0.15\\cdot\\overline{\\text{PAS}}$$

---

### 6.3 Figure Caption Templates

- **Fig. 1** — Heatmap of mean action hallucination rate per model and domain. Cells annotated with the fraction of plan actions whose names are absent from the legal schema set.
- **Fig. 2** — Failure mode taxonomy plot. Each point represents a model positioned by hallucination rate (x-axis) and Precondition Awareness Score (y-axis). Quadrants identify distinct failure strategies.
- **Fig. 3** — Composite Planning Score ranked bar chart with 95% bootstrap confidence intervals (n=1000 resamples). Stacked bars decompose the total score into per-domain contributions.
- **Fig. 4** — Radar chart showing the five PS component metrics per model, enabling holistic comparison of planning capability profiles.
- **Fig. 5** — Per-iteration conditional success probability P(Valid | reached attempt k). A flat curve suggests random resampling; a rising curve indicates genuine iterative repair.

---

### 6.4 Methods Paragraph (for paper)

> Advanced metrics were computed from a custom lightweight PDDL simulator that tracks both
> propositional state (ground atoms) and numeric fluent values across plan steps.  For each plan,
> we ground action schemas using the arguments provided by the model, evaluate preconditions
> against the current state, and advance the simulation if and only if all conditions are
> satisfied.  The first step at which execution fails defines the *executability prefix*.
> Failed propositional preconditions are classified as *sequencing errors* if the required
> atom held in any earlier state (temporal distance $\\delta > 0$) or as *state fabrications*
> otherwise.  The Precondition Awareness Score (PAS) is the fraction of failures that are
> sequencing rather than fabrication errors, reflecting whether the model has implicit causal
> knowledge despite incorrect ordering.
> Action hallucination is detected by exact match and separately by fuzzy match
> (Levenshtein distance $\\leq 2$) of action names against the domain schema set.  Object
> hallucination is detected by checking plan arguments against the problem object set.
> CoT–plan alignment is computed for runs with Chain-of-Thought enabled by measuring the
> overlap between tokens in the reasoning text and the action/object vocabularies used in
> the final plan.